# Case Study - Group 16: OEM1 Emissions Investigation

## Objective

The objective of this case study is to identify all OEM1 vehicles affected by the potentially defective T2 control units and determine the municipality in which these vehicles were registered.

According to the investigation information, affected T2 control units were produced by manufacturer 202 in plant 2022 between April 2009 and November 2014.

In addition, control units produced by manufacturer 201 in plant 2011 are affected when their production numbers range from 1250 to 19500.

The affected T2 control units are installed in OEM1 engines. These engines can be installed in OEM1 Type11 and Type12 vehicles.

The analysis therefore follows the supply chain:

**T2 control units → K1 engine components → OEM1 Type11/Type12 vehicles → vehicle registrations → municipality**

The available data are inspected and combined to create the final set of affected registered vehicles.

## 1. Data Selection

The available database contains information about individual parts, components, vehicles, registrations, geodata and logistics.

For this case study, the relevant datasets are:

- `Einzelteil_T02.txt` – production data for T2 control units
- `Bestandteile_Komponente_K1BE1.csv`
- `Bestandteile_Komponente_K1BE2.csv`
- `Bestandteile_Komponente_K1DI1.csv`
- `Bestandteile_Komponente_K1DI2.csv`
- `Bestandteile_Fahrzeuge_OEM1_Typ11.csv`
- `Bestandteile_Fahrzeuge_OEM1_Typ12.csv`
- `Zulassungen_alle_Fahrzeuge.csv`

The K1 component tables are required because they contain the relationship between T2 control units and the K1 engine components.

The OEM1 vehicle tables are required because they contain the relationship between the engine component and the vehicle.

Finally, the registration table is required to determine the municipality in which an affected vehicle was registered.

## 2. Import and Prepare the T2 Data

The original `Einzelteil_T02.txt` file contains records separated by tab characters.

The checkpoint analysis showed that the file contains two sets of T2 records. To make the file readable as a normal table, the tab characters are replaced by line breaks and the resulting file is read using whitespace separation.

The resulting dataset contains the T2 identifier, production date, manufacturer, production plant and defect information.

In [ ]:
import pandas as pd
from pathlib import Path

data_path = Path("data/IDA SoSe26 - Data")

input_file = (data_path / "Einzelteil" / "Einzelteil_T02.txt")
fixed_file = (data_path / "Einzelteil" / "Einzelteil_T02_fixed.txt")
print(f"Input file: {input_file}")
print(f"Fixed file: {fixed_file}")

with open(input_file, "r", encoding="utf-8") as src, \
     open(fixed_file, "w", encoding="utf-8") as dst:

    while chunk := src.read(10_000_000):
        dst.write(chunk.replace("\t", "\n"))

df_t02 = pd.read_csv(
    fixed_file,
    sep=r"\s+",
    quotechar='"',
    na_values="NA"
)

print("Shape:", df_t02.shape)
print(df_t02.columns.tolist())
display(df_t02.head())

## 3. Create a Unified T2 Dataset

The original T2 data contain two sets of columns (`.x` and `.y`).

The two sets are combined using `combine_first()` so that the final dataset contains one consistent set of T2 attributes.

This produces one row per T2 control unit with:

- T2 ID
- production date
- manufacturer
- production plant
- defect information
- defect date
- mileage at defect

In [ ]:
t02 = pd.DataFrame({
    "ID_T02": df_t02["ID_T02.x"].combine_first(df_t02["ID_T02.y"]),
    "Produktionsdatum": df_t02["Produktionsdatum.x"].combine_first(
        df_t02["Produktionsdatum.y"]
    ),
    "Herstellernummer": df_t02["Herstellernummer.x"].combine_first(
        df_t02["Herstellernummer.y"]
    ),
    "Werksnummer": df_t02["Werksnummer.x"].combine_first(
        df_t02["Werksnummer.y"]
    ),
    "Fehlerhaft": df_t02["Fehlerhaft.x"].combine_first(
        df_t02["Fehlerhaft.y"]
    ),
    "Fehlerhaft_Datum": df_t02["Fehlerhaft_Datum.x"].combine_first(
        df_t02["Fehlerhaft_Datum.y"]
    ),
    "Fehlerhaft_Fahrleistung": df_t02[
        "Fehlerhaft_Fahrleistung.x"
    ].combine_first(
        df_t02["Fehlerhaft_Fahrleistung.y"]
    )
})

print(t02.shape)
display(t02.head())

## 4. Identify Affected T2 Control Units

Two groups of T2 control units are affected according to the case description.

### Group 1

Control units produced by:

- Manufacturer: **202**
- Plant: **2022**
- Production period: **April 2009 to November 2014**

### Group 2

Control units produced by:

- Manufacturer: **201**
- Plant: **2011**
- Production number: **1250 to 19500**

Both groups are identified separately and then combined into one affected T2 dataset.

In [ ]:
t02_202 = t02[
    (t02["Herstellernummer"] == 202) &
    (t02["Werksnummer"] == 2022)
].copy()

t02_202["Produktionsdatum"] = pd.to_datetime(
    t02_202["Produktionsdatum"]
)

affected_202 = t02_202[
    (t02_202["Produktionsdatum"] >= "2009-04-01") &
    (t02_202["Produktionsdatum"] < "2014-12-01")
].copy()

print("Affected 202/2022:", affected_202.shape)

In [ ]:
t02_201 = t02[
    (t02["Herstellernummer"] == 201) &
    (t02["Werksnummer"] == 2011)
].copy()

t02_201["Produktionsnummer"] = (
    t02_201["ID_T02"]
    .str.rsplit("-", n=1)
    .str[-1]
    .astype(int)
)

affected_201 = t02_201[
    t02_201["Produktionsnummer"].between(1250, 19500)
].copy()

print("Affected 201/2011:", affected_201.shape)

In [ ]:
affected_t2 = pd.concat(
    [affected_202, affected_201],
    ignore_index=True
)

print("Affected T2 units:", affected_t2.shape)
print("Unique T2 IDs:", affected_t2["ID_T02"].nunique())
print(
    "Duplicate T2 IDs:",
    affected_t2["ID_T02"].duplicated().sum()
)

## 5. Trace Affected T2 Units to K1 Engine Components

The affected T2 units are installed in K1 engine components.

Per the general case-study brief, affected T2 units were installed in
Gasoline engines. We therefore restrict this analysis to the gasoline engine components:

- K1BE1 (Benzin / Gasoline)
- K1BE2 (Benzin / Gasoline)

(We discard following diesel engines components: K1DI1, K1DI2)

Each table contains an `ID_T2` column and the corresponding K1 component ID.

The affected T2 IDs are therefore used to filter each K1 dataset.

The resulting relationships are combined into one mapping table between affected T2 units and K1 engine components.

Restricting columns on read keeps memory usage down given these files are 100MB+.

In [ ]:
K1BE1_file = (data_path / "Komponente" / "Bestandteile_Komponente_K1BE1.csv")
K1BE2_file = (data_path / "Komponente" / "Bestandteile_Komponente_K1BE2.csv")

affected_t2_ids = set(affected_t2["ID_T02"])

def load_k1_affected(path, id_col, k1_type):
    """Load a K1 component parts-list file, restricted to columns needed
    for the T2 > K1 join, and filter to rows containing an affected T2."""
    df = pd.read_csv(
        path,
        sep=";",
        quotechar='"',
        usecols=["ID_T2", id_col],
        dtype=str
    )
    df = df[df["ID_T2"].isin(affected_t2_ids)].copy()
    df = df.rename(columns={id_col: "ID_K1"})
    df["K1_Type"] = k1_type
    return df[["ID_T2", "ID_K1", "K1_Type"]]

k1be1_affected = load_k1_affected(
    K1BE1_file, "ID_K1BE1", "K1BE1"
)
k1be2_affected = load_k1_affected(
    K1BE2_file, "ID_K1BE2", "K1BE2"
)

affected_t2_k1 = pd.concat(
    [k1be1_affected, k1be2_affected],
    ignore_index=True
)

print("Affected T2 > K1 mappings:", affected_t2_k1.shape)
print(affected_t2_k1["K1_Type"].value_counts())
print("Unique T2 matched:", affected_t2_k1["ID_T2"].nunique())
print("Duplicate T2 matches:", affected_t2_k1["ID_T2"].duplicated().sum())
display(affected_t2_k1.head())

## 6. Identify Affected OEM1 Vehicles

The OEM1 vehicle parts lists contain the engine component installed in each vehicle in the column `ID_Motor`.

Because the affected K1 components are the affected engine components, vehicles are identified by checking whether their `ID_Motor` occurs in the set of affected K1 IDs.

The analysis is performed separately for:

- OEM1 Type11
- OEM1 Type12

The two vehicle datasets are then combined.

In [ ]:
type11_file = (data_path / "Fahrzeug" / "Bestandteile_Fahrzeuge_OEM1_Typ11.csv")
type12_file = (data_path / "Fahrzeug" / "Bestandteile_Fahrzeuge_OEM1_Typ12.csv")

vehicles_11 = pd.read_csv(
    type11_file,
    sep=";",
    quotechar='"',
    usecols=["ID_Motor", "ID_Fahrzeug"],
    dtype=str
)

vehicles_12 = pd.read_csv(
    type12_file,
    sep=";",
    quotechar='"',
    usecols=["ID_Motor", "ID_Fahrzeug"],
    dtype=str
)

affected_vehicles = pd.concat(
    [vehicles_11, vehicles_12],
    ignore_index=True
)

affected_k1_ids = set(affected_t2_k1["ID_K1"])

affected_vehicles_11 = vehicles_11[
    vehicles_11["ID_Motor"].isin(affected_k1_ids)
].copy()
affected_vehicles_11["Vehicle_Type"] = "Type11"

affected_vehicles_12 = vehicles_12[
    vehicles_12["ID_Motor"].isin(affected_k1_ids)
].copy()
affected_vehicles_12["Vehicle_Type"] = "Type12"

affected_vehicles = pd.concat(
    [affected_vehicles_11, affected_vehicles_12],
    ignore_index=True
)

print("Affected Type11 vehicles:", affected_vehicles_11.shape)
print("Affected Type12 vehicles:", affected_vehicles_12.shape)
print("Total affected vehicles:", affected_vehicles.shape)
print("Unique vehicle IDs:", affected_vehicles["ID_Fahrzeug"].nunique())
print("Duplicate vehicle IDs:", affected_vehicles["ID_Fahrzeug"].duplicated().sum())
display(affected_vehicles.head())

## 7. Link Affected Vehicles to Registration Data

The registration dataset contains:

- `IDNummer` – vehicle identifier
- `Gemeinden` – municipality
- `Zulassung` – registration date

The vehicle identifier in the production data is `ID_Fahrzeug`.

Therefore, the registration data are linked using:

`ID_Fahrzeug = IDNummer`

The registration table contains unique vehicle identifiers, so a one-to-one merge is expected.

In [ ]:
registrations_file = data_path / "Zulassungen" / "Zulassungen_alle_Fahrzeuge.csv"

registrations = pd.read_csv(
    registrations_file,
    sep=";",
    quotechar='"'
)

print("Registration rows:", len(registrations))
print("Unique IDNummer:", registrations["IDNummer"].nunique())
print("Duplicate IDNummer:",registrations["IDNummer"].duplicated().sum())
display(registrations.head())

In [ ]:
registrations_small = registrations[
    ["IDNummer", "Gemeinden", "Zulassung"]
].copy()

try:
    affected_registered = affected_vehicles.merge(
        registrations_small,
        left_on="ID_Fahrzeug",
        right_on="IDNummer",
        how="left",
        validate="one_to_one"
    )
except Exception as e:
    print("Merge validation FAILED:", e)
    raise

print(affected_registered.shape)
display(affected_registered.head())

In [ ]:
print("affected_vehicles shape:", affected_vehicles.shape)
print(affected_vehicles["Vehicle_Type"].value_counts())
print()

# Break down by which K1 family the matched engine came from
k1_type_lookup = affected_t2_k1.set_index("ID_K1")["K1_Type"]
affected_vehicles["K1_Type"] = affected_vehicles["ID_Motor"].map(k1_type_lookup)
print(affected_vehicles["K1_Type"].value_counts(dropna=False))

**Note on K1BE2:** Although K1BE2 engine components were included in the T2 → K1 mapping (Step 5) for completeness, verification shows that OEM1 \
Type11 and Type12 vehicles exclusively use K1BE1 (gasoline) or K1DI1 (diesel) engines — K1BE2 and K1DI2 do not appear in either vehicle type's \
parts list. K1BE2 is therefore expected to contribute zero matches to the final affected-vehicle count, which is confirmed empirically below. 

In [ ]:
# What do affected K1BE2 engine IDs actually look like?
k1be2_ids_affected = affected_t2_k1[affected_t2_k1["K1_Type"] == "K1BE2"]["ID_K1"]
print("Sample affected K1BE2 IDs:")
print(k1be2_ids_affected.head(10).tolist())
print()

# What ID_Motor values actually exist in the vehicle files, for comparison?
print("Sample ID_Motor values in Typ11:")
print(vehicles_11["ID_Motor"].head(10).tolist())
print()

# Direct check: does ANY K1BE2 engine (affected or not) ever appear as an ID_Motor
# in either vehicle file? This tests independent of the "affected" filter.
all_motor_ids_11 = set(vehicles_11["ID_Motor"])
all_motor_ids_12 = set(vehicles_12["ID_Motor"])
all_motor_ids = all_motor_ids_11 | all_motor_ids_12

k1be2_prefix_matches = [m for m in list(all_motor_ids)[:50000] if m.startswith("K1BE2")]
print("Sample of any K1BE2-prefixed engine IDs found in vehicle files:", k1be2_prefix_matches[:10])
print("Count found in this 50k sample:", len(k1be2_prefix_matches))

## 8. Validate the Registration Merge

The registration merge is checked for vehicles for which no municipality was found.

A successful merge should result in no affected vehicles with a missing municipality.

In [ ]:
print(
    "Affected vehicles without registration:",
    affected_registered["Gemeinden"].isna().sum()
)

print(
    "Affected vehicles with registration:",
    affected_registered["Gemeinden"].notna().sum()
)

## 9. Affected Vehicles by Municipality

The final registered vehicle dataset can now be aggregated by municipality.

This provides the number of affected OEM1 vehicles registered in each municipality.

In [ ]:
municipality_counts = (
    affected_registered["Gemeinden"]
    .value_counts()
    .rename_axis("Gemeinde")
    .reset_index(name="Affected_Vehicles")
)
print("Number of municipalities:", len(municipality_counts))
print(municipality_counts.head(20))

In [ ]:
# Check for any municipality names with a trailing digit — possible
# artifact of a deduplication or export process upstream.
suspicious = registrations["Gemeinden"].str.contains(r"\d$", regex=True, na=False)
print("Municipalities with trailing digits:")
print(registrations.loc[suspicious, "Gemeinden"].value_counts())
print()

# Compare directly: how many total registrations for MUENSTER vs MUENSTER1?
print(registrations["Gemeinden"].value_counts().loc[["MUENSTER", "MUENSTER1"]])

In [ ]:
geodata = pd.read_csv(
    data_path / "Geodaten" / "Geodaten_Gemeinden_v1.2_2017-08-22_TrR.csv",
    sep=";", quotechar='"'
)

print("Geodaten columns:", geodata.columns.tolist())

In [ ]:
geodata_small = geodata[["Gemeinde", "Postleitzahl", "Laengengrad", "Breitengrad"]].copy()

final_dataset = affected_registered.merge(
    geodata_small,
    left_on="Gemeinden",
    right_on="Gemeinde",
    how="left"
)

print(final_dataset.shape)
print("Rows without matched geodata:", final_dataset["Gemeinde"].isna().sum())
display(final_dataset.head())

In [ ]:
# For each unmatched name, check for near-matches in geodata
# (e.g., trailing whitespace, different suffix, alternate spelling)
unmatched = final_dataset[final_dataset["Gemeinde"].isna()]
print(unmatched["Gemeinden"].value_counts())
unmatched_names = unmatched["Gemeinden"].unique()
print("Unmatched municipality names:", unmatched_names)
print()

for name in unmatched_names:
    print(f"--- '{name}' ---")
    # exact match check (repr shows hidden whitespace)
    print("repr:", repr(name))
    # substring match in geodata, in case of partial name differences
    close = geodata[geodata["Gemeinde"].str.contains(name.strip(), case=False, na=False, regex=False)]
    print(close[["Gemeinde", "Postleitzahl"]].to_string())
    print()

**Note on unmatched geodata:** 36 of 597,655 affected, registered vehicles (0.006%) are registered in `SEEG`, a municipality not present in the \
provided geodata reference table (`Geodaten_Gemeinden_v1.2_2017-08-22`). These rows are retained in the final dataset with missing coordinate \
values, and are excluded only from the app's map visualization — they remain fully represented in the tabular data view and all non-geographic \
analysis.

In [ ]:
output_path = "data/SoSe26_Case_Study_finalData_Group_16.csv"

final_dataset.to_csv(output_path, index=False)
print("Saved:", output_path)
print("Final shape:", final_dataset.shape)
display(final_dataset.head())